## 05 — Persona Insights

Interpreting clusters as business personas with actionable recommendations.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import PERSONA_MAP, PERSONA_DESCRIPTIONS, BUSINESS_RECOMMENDATIONS, REPORTS_DIR

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

### Load Clustered Data

In [2]:
from src.config import PROCESSED_FILES

clustered = pd.read_csv(PROCESSED_FILES["with_clusters"]) if PROCESSED_FILES["with_clusters"].exists() else None
if clustered is not None:
    print(f"Loaded: {clustered.shape}")
    print(clustered.head())
else:
    print("Run the pipeline first: python -m src.pipeline")

Loaded: (1000, 14)
  CustomerID   Recency  Frequency  Monetary  PurchaseInterval  WeekendRatio  \
0      C1000 -0.077838   1.908854  3.183311         -0.971415     -1.479184   
1      C1001 -0.565932  -0.318142 -1.004300          0.407924      0.316546   
2      C1002 -0.376118   0.636285  0.276546         -0.504166      0.316546   
3      C1003  0.613630  -0.318142 -0.549529         -0.095663     -1.156873   
4      C1004 -0.904887   0.318142 -1.046939         -0.167972      2.325754   

   NightRatio  DiscountUsage  ReturnRate  ProductDiversity  AvgQuantity  \
0    0.405247      -1.076420   -0.540076          1.951147     3.500000   
1   -0.191237       1.308555   -0.641915         -0.302653     2.777778   
2    0.542897       1.308555    0.376476          0.341290     3.083333   
3   -0.191237       0.460564   -0.641915         -0.302653     2.333333   
4    0.743115      -1.466689   -0.790045          0.341290     2.181818   

   Cluster       PC1       PC2  
0        2  3.893204  

### Persona Profiles

In [3]:
if clustered is not None:
    cluster_col = [c for c in clustered.columns if "cluster" in c.lower()][0]
    profiles = clustered.groupby(cluster_col).mean(numeric_only=True)
    display(profiles)

,Recency,Frequency,Monetary,PurchaseInterval,WeekendRatio,NightRatio,DiscountUsage,ReturnRate,ProductDiversity,AvgQuantity,PC1,PC2
Cluster,,,,,,,,,,,,
0,-0.235743,-0.113165,-0.196021,-0.061228,0.053059,-0.113052,0.079651,-0.021895,-0.114656,2.979763,-0.130682,-0.066511
1,2.125619,-0.745278,-0.410128,-0.208361,-0.274193,-0.034121,-0.350175,0.144448,-0.746855,2.930173,-1.318980,-0.575760
2,-0.409818,1.224505,1.006120,-0.780094,-0.023376,0.074405,0.054795,-0.017507,1.227290,3.023271,2.180362,0.079100
3,-0.061438,-1.253519,-0.884948,1.597234,0.074217,0.200809,-0.077954,-0.005669,-1.253144,2.889307,-2.426802,0.418698


### Business Recommendations

In [4]:
print("Persona Recommendations:")
print("=" * 60)
for persona, recs in BUSINESS_RECOMMENDATIONS.items():
    print(f"\n{persona}:")
    for r in recs:
        print(f"  - {r}")

Persona Recommendations:

VIP Loyal Customers:
  - Exclusive rewards & VIP loyalty program
  - Early access to new products
  - Personalized recommendations

Discount Hunters:
  - Targeted coupon campaigns
  - Flash sale notifications
  - Bundle deals to increase basket size

Churn Risk:
  - Win-back email campaigns
  - Special reactivation discounts
  - Survey to understand inactivity

One-Time Buyers:
  - Cross-selling on complementary products
  - Post-purchase follow-up sequence
  - Free shipping on second purchase


### Export Summary

In [5]:
import csv
import os

os.makedirs(REPORTS_DIR, exist_ok=True)
summary_path = REPORTS_DIR / "persona_summary.csv"

rows = []
for cid, persona in PERSONA_MAP.items():
    rows.append({"Cluster": cid, "Persona": persona, "Description": PERSONA_DESCRIPTIONS.get(persona, "")})

with open(summary_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["Cluster", "Persona", "Description"])
    w.writeheader()
    w.writerows(rows)

print(f"Exported: {summary_path}")

Exported: D:\dr\buyer-persona-ml\reports\persona_summary.csv
